In [1]:
import optuna
import axelrod
from axelrod.action import Action, actions_to_str
from axelrod.player import Player
from axelrod.strategy_transformers import (
    FinalTransformer,
    TrackHistoryTransformer,
)
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import numpy as np
from collections import Counter
import pandas as pd
from math import exp
import numpy as np


c:\Users\Ognjen\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
C, D = Action.C, Action.D

class FuzzyMethods():
    @staticmethod
    def calc_cooperation(self, opponent):
        return (Counter(opponent.history)[C])/len(opponent.history)*100
    
    @staticmethod
    def calc_adaptivity(self, opponent):
        adapCounter = 0
        adapReaction = 0

        if(len(self.history) < 3): 
            return 0
        
        for i in range(3, len(self.history)):
            if (self.history[i-3] == C and self.history[i-2] == D):
                adapCounter += 1
                if (opponent.history[i-1] == D):
                    adapReaction += 1
                elif (self.history[i-1] == D and opponent.history[i] == D):
                    adapReaction += 0.5
            elif (self.history[i-3] == D and self.history[i-2] == C):
                adapCounter += 1
                if (opponent.history[i-1] == C):
                    adapReaction += 1
                elif (self.history[i-1] == C and opponent.history[i] == C):
                    adapReaction += 0.5

        if adapCounter == 0:
            return 0
        
        return adapReaction/adapCounter*100

    
    @staticmethod
    def calc_forgiveness(self, opponent):
        DCounter = 0
        punishmentCounter = 0

        for i in range(0, len(self.history)-1):
            if(self.history[i] == D):
                DCounter += 1
                for j in range (i+1, len(opponent.history)):
                    if(opponent.history[j] == C):
                        break
                    else:
                        punishmentCounter += 1
              

        if punishmentCounter > 0:
            return DCounter/punishmentCounter*100
        else:
            return 100
    
    @staticmethod
    def calc_stochastic(self, opponent):
        patterns = [
            [C, C, C],
            [C, C, D],
            [C, D, C],
            [C, D, D],
            [D, C, C],
            [D, C, D],
            [D, D, C],
            [D, D, D]
        ]

        non_stochasticCounter = 0
        patternPlayedCounter = 0

        for p in patterns:
            opponentsReactions = []
            for i in range(0, len(self.history)-3):
                if ([self.history[i], self.history[i+1], self.history[i+2]] == p):
                    opponentsReactions.append([opponent.history[i+1], opponent.history[i+2], opponent.history[i+3]])
            
            unique_patterns = len(set(tuple(sub) for sub in opponentsReactions))

            if(len(opponentsReactions) > 0):
                non_stochasticCounter += 0 if unique_patterns == 1 else unique_patterns
                patternPlayedCounter += len(opponentsReactions)
        
        if patternPlayedCounter == 0:
            return 0
        
        return non_stochasticCounter/patternPlayedCounter*100
    

In [ ]:
def build_fuzzy_player(params):
    _cooperation = ctrl.Antecedent(np.arange(0, 100, 1), 'cooperation')
    _adaptivity  = ctrl.Antecedent(np.arange(0, 100, 1), 'adaptivity')
    _forgiveness = ctrl.Antecedent(np.arange(0, 100, 1), 'forgiveness')
    _stochastic  = ctrl.Antecedent(np.arange(0, 100, 1), 'stochastic')
    _cooperation['low']    = fuzz.trimf(_cooperation.universe, [
        params['coop_low_a'], params['coop_low_b'], params['coop_low_c']
    ])
    _cooperation['medium'] = fuzz.trimf(_cooperation.universe, [
        params['coop_med_a'], params['coop_med_b'], params['coop_med_c']
    ])
    _cooperation['high']   = fuzz.trimf(_cooperation.universe, [
        params['coop_high_a'], params['coop_high_b'], params['coop_high_c']
    ])

    # --- _adaptivity (no, yes) ---
    _adaptivity['no']  = fuzz.trimf(_adaptivity.universe, [
        params['adap_no_a'], params['adap_no_b'], params['adap_no_c']
    ])
    _adaptivity['yes'] = fuzz.trimf(_adaptivity.universe, [
        params['adap_yes_a'], params['adap_yes_b'], params['adap_yes_c']
    ])

    # --- _forgiveness (low, medium, high) ---
    _forgiveness['low']    = fuzz.gaussmf(_forgiveness.universe, 0, params['forg_sigma'])
    _forgiveness['medium'] = fuzz.trimf(_forgiveness.universe, [
        params['forg_med_a'], params['forg_med_b'], params['forg_med_c']
    ])
    _forgiveness['high']   = fuzz.trimf(_forgiveness.universe, [
        params['forg_high_a'], params['forg_high_b'], params['forg_high_c']
    ])

    # --- _stochastic (none, sometimes, always) ---
    _stochastic['none']      = fuzz.trimf(_stochastic.universe, [
        params['stoch_none_a'], params['stoch_none_b'], params['stoch_none_c']
    ])
    _stochastic['sometimes'] = fuzz.trimf(_stochastic.universe, [
        params['stoch_some_a'], params['stoch_some_b'], params['stoch_some_c']
    ])
    _stochastic['always']    = fuzz.trimf(_stochastic.universe, [
        params['stoch_alw_a'], params['stoch_alw_b'], params['stoch_alw_c']
    ])

    # Resulting strategy MFs — also being optimized
    _resulting_strategy = ctrl.Consequent(np.arange(0, 100, 1), 'resulting_strategy')
    _resulting_strategy['D'] = fuzz.trimf(_resulting_strategy.universe, [
        params['D_a'],
        params['D_b'],
        params['D_c']
    ])
    _resulting_strategy['C'] = fuzz.trimf(_resulting_strategy.universe, [
        params['C_a'],
        params['C_b'],
        params['C_c']
    ])

    # Rebuild rules using the fresh variables above
    rule1 = ctrl.Rule(
        _cooperation['high'] & _adaptivity['no'] & (_forgiveness['medium'] | _forgiveness['high']),
        _resulting_strategy['D']
    )
    rule2 = ctrl.Rule(
        _forgiveness['low'] & _cooperation['high'],
        _resulting_strategy['C']
    )
    rule3 = ctrl.Rule(
        _stochastic['always'] | _adaptivity['no'],
        _resulting_strategy['D']
    )
    rule4 = ctrl.Rule(
        _cooperation['low'] | (_cooperation['medium'] & _forgiveness['low']),
        _resulting_strategy['D']
    )
    rule5 = ctrl.Rule(
        _cooperation['medium'] & _forgiveness['medium'] & _adaptivity['yes'],
        _resulting_strategy['C']
    )

    strategy_ctrl  = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5])
    _chosen_strategy = ctrl.ControlSystemSimulation(strategy_ctrl)

    # Build the player class dynamically, capturing everything in closure
    class OptimizedFuzzy(Player):

        # Override class-level FIS components with the fresh ones
        cooperation = _cooperation
        adaptivity = _adaptivity
        stochastic = _stochastic
        forgiveness = _forgiveness
        resulting_strategy = _resulting_strategy
        chosen_strategy = _chosen_strategy

        d_thresh = params['d_threshold']
        c_thresh = params['c_threshold']

        # Reset state so trials don't bleed into each other
        first_time = True
        h = {'Name': '', 'Fuzzy': [], 'Opponent': []}

        def strategy(self, opponent: axelrod.Player) -> Action:

            if len(self.history) == 0 or D not in opponent.history:
                return C

            coop  = FuzzyMethods.calc_cooperation(self, opponent)
            adap  = FuzzyMethods.calc_adaptivity(self, opponent)
            forg  = FuzzyMethods.calc_forgiveness(self, opponent)
            stoch = FuzzyMethods.calc_stochastic(self, opponent)

            self.chosen_strategy.input['cooperation'] = coop
            self.chosen_strategy.input['adaptivity']  = adap
            self.chosen_strategy.input['forgiveness'] = forg
            self.chosen_strategy.input['stochastic']  = stoch

            try:
                self.chosen_strategy.compute()
                output_val = self.chosen_strategy.output['resulting_strategy']
            except KeyError:
                # No rules fired — default to cooperate
                return C
            except Exception:
                return C

            d_membership = fuzz.interp_membership(
                self.resulting_strategy.universe,
                self.resulting_strategy['D'].mf,
                output_val
            )
            c_membership = fuzz.interp_membership(
                self.resulting_strategy.universe,
                self.resulting_strategy['C'].mf,
                output_val
            )

            if d_membership >= self.d_thresh and c_membership < self.c_thresh:
                return D

            return C

    return OptimizedFuzzy()

In [ ]:
def sample_trimf(trial, name, universe_min, universe_max):
    """Sample a, b, c such that a <= b <= c is always guaranteed."""
    a = trial.suggest_int(f'{name}_a', universe_min, universe_max - 2)
    b = trial.suggest_int(f'{name}_b', a, universe_max - 1)
    c = trial.suggest_int(f'{name}_c', b, universe_max)
    return a, b, c

params = {

    # COOPERATION — automf(["low", "medium", "high"])
    'coop_low_a':  0,  'coop_low_b':  0,  'coop_low_c':  49,
    'coop_med_a':  0,  'coop_med_b':  49, 'coop_med_c':  99,
    'coop_high_a': 49, 'coop_high_b': 99, 'coop_high_c': 99,

    # ADAPTIVITY — automf(["no", "yes"])
    'adap_no_a':   0,  'adap_no_b':  0,  'adap_no_c':  99,
    'adap_yes_a':  0,  'adap_yes_b': 99, 'adap_yes_c': 99,

    # FORGIVENESS — gaussmf sigma=25, trimf medium, automf high
    'forg_sigma':   25,
    'forg_med_a':   25, 'forg_med_b':  50, 'forg_med_c':  75,
    'forg_high_a':  49, 'forg_high_b': 99, 'forg_high_c': 99,

    # STOCHASTIC — automf(["none", "sometimes", "always"])
    'stoch_none_a': 0,  'stoch_none_b': 0,  'stoch_none_c': 49,
    'stoch_some_a': 0,  'stoch_some_b': 49, 'stoch_some_c': 99,
    'stoch_alw_a':  49, 'stoch_alw_b':  99, 'stoch_alw_c':  99,

    # OUTPUT
    'D_a': 0,  'D_b': 25, 'D_c': 50,
    'C_a': 35, 'C_b': 75, 'C_c': 99,   # your 100 clamped to 99 (universe ends at 99)

    # THRESHOLDS
    'd_threshold': 0.4,
    'c_threshold': 0.6,
}

def objective(trial):

    # --- COOPERATION ---
    coop_low_a,  coop_low_b,  coop_low_c  = sample_trimf(trial, 'coop_low',   0, 50)
    coop_med_a,  coop_med_b,  coop_med_c  = sample_trimf(trial, 'coop_med',  15, 80)
    coop_high_a, coop_high_b, coop_high_c = sample_trimf(trial, 'coop_high', 50, 99)

    # --- ADAPTIVITY ---
    adap_no_a,  adap_no_b,  adap_no_c  = sample_trimf(trial, 'adap_no',   0, 60)
    adap_yes_a, adap_yes_b, adap_yes_c = sample_trimf(trial, 'adap_yes', 40, 99)

    # --- FORGIVENESS ---
    forg_sigma                          = trial.suggest_float('forg_sigma', 5, 40)
    forg_med_a,  forg_med_b,  forg_med_c  = sample_trimf(trial, 'forg_med',  10, 90)
    forg_high_a, forg_high_b, forg_high_c = sample_trimf(trial, 'forg_high', 60, 99)

    # --- STOCHASTIC ---
    stoch_none_a, stoch_none_b, stoch_none_c = sample_trimf(trial, 'stoch_none',  0, 50)
    stoch_some_a, stoch_some_b, stoch_some_c = sample_trimf(trial, 'stoch_some', 20, 80)
    stoch_alw_a,  stoch_alw_b,  stoch_alw_c  = sample_trimf(trial, 'stoch_alw',  55, 99)

    # --- OUTPUT ---
    D_a, D_b, D_c = sample_trimf(trial, 'D',  0, 60)
    C_a, C_b, C_c = sample_trimf(trial, 'C', 25, 99)

    # --- THRESHOLDS ---
    d_threshold = trial.suggest_float('d_threshold', 0.2, 0.7)
    c_threshold = trial.suggest_float('c_threshold', 0.3, 0.8)

    

    try:
        fuzzy_player = build_fuzzy_player(params)
    except Exception as e:
        print(f"Failed to build player: {e}")
        return 0.0

    opponents = [player() for player in axelrod.stewart_plotkin_strategies]

    try:
        tournament = axelrod.Tournament(
            [fuzzy_player] + opponents,
            turns=200,
        )
        results = tournament.play(progress_bar=False)
    except Exception as e:
        print(f"Tournament failed: {e}")
        return 0.0

    return np.mean(results.normalised_scores[0])

In [ ]:
study = optuna.create_study(
    direction='maximize',
    study_name='fuzzy_optimization_full_start',
    storage='sqlite:///fuzzy_optuna_full_start.db',
    load_if_exists=True
)
study.enqueue_trial(params)
study.optimize(objective, n_trials=300, show_progress_bar=True)

print("\n=== OPTIMIZATION COMPLETE ===")
print(f"Best score:  {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

importance = optuna.importance.get_param_importances(study)
print("\n=== PARAMETER IMPORTANCE ===")
for param, imp in importance.items():
    print(f"  {param}: {imp:.4f}")

[I 2026-03-03 16:39:57,004] Using an existing study with name 'fuzzy_optimization_full_start' instead of creating a new one.
  0%|          | 0/300 [00:00<?, ?it/s]c:\Users\Ognjen\AppData\Local\Programs\Python\Python311\Lib\site-packages\optuna\trial\_trial.py:658: UserWarning: Fixed parameter 'coop_med_a' with value 0 is out of range for distribution IntDistribution(high=78, log=False, low=15, step=1).
  optuna_warn(
c:\Users\Ognjen\AppData\Local\Programs\Python\Python311\Lib\site-packages\optuna\trial\_trial.py:658: UserWarning: Fixed parameter 'coop_med_c' with value 99 is out of range for distribution IntDistribution(high=80, log=False, low=49, step=1).
  optuna_warn(
c:\Users\Ognjen\AppData\Local\Programs\Python\Python311\Lib\site-packages\optuna\trial\_trial.py:658: UserWarning: Fixed parameter 'coop_high_a' with value 49 is out of range for distribution IntDistribution(high=97, log=False, low=50, step=1).
  optuna_warn(
c:\Users\Ognjen\AppData\Local\Programs\Python\Python311\Lib

[I 2026-03-03 16:40:25,283] Trial 1 finished with value: 2.705964285714286 and parameters: {'coop_low_a': 0, 'coop_low_b': 0, 'coop_low_c': 49, 'coop_med_a': 0, 'coop_med_b': 49, 'coop_med_c': 99, 'coop_high_a': 49, 'coop_high_b': 99, 'coop_high_c': 99, 'adap_no_a': 0, 'adap_no_b': 0, 'adap_no_c': 99, 'adap_yes_a': 0, 'adap_yes_b': 99, 'adap_yes_c': 99, 'forg_sigma': 25, 'forg_med_a': 25, 'forg_med_b': 50, 'forg_med_c': 75, 'forg_high_a': 49, 'forg_high_b': 99, 'forg_high_c': 99, 'stoch_none_a': 0, 'stoch_none_b': 0, 'stoch_none_c': 49, 'stoch_some_a': 0, 'stoch_some_b': 49, 'stoch_some_c': 99, 'stoch_alw_a': 49, 'stoch_alw_b': 99, 'stoch_alw_c': 99, 'D_a': 0, 'D_b': 25, 'D_c': 50, 'C_a': 35, 'C_b': 75, 'C_c': 99, 'd_threshold': 0.4, 'c_threshold': 0.6}. Best is trial 1 with value: 2.705964285714286.


Best trial: 1. Best value: 2.70596:   1%|          | 2/300 [01:03<2:41:58, 32.61s/it]

[I 2026-03-03 16:41:00,945] Trial 2 finished with value: 2.6997142857142857 and parameters: {'coop_low_a': 17, 'coop_low_b': 23, 'coop_low_c': 29, 'coop_med_a': 48, 'coop_med_b': 73, 'coop_med_c': 80, 'coop_high_a': 82, 'coop_high_b': 87, 'coop_high_c': 87, 'adap_no_a': 38, 'adap_no_b': 47, 'adap_no_c': 48, 'adap_yes_a': 85, 'adap_yes_b': 86, 'adap_yes_c': 86, 'forg_sigma': 8.10206420346769, 'forg_med_a': 33, 'forg_med_b': 59, 'forg_med_c': 77, 'forg_high_a': 71, 'forg_high_b': 82, 'forg_high_c': 86, 'stoch_none_a': 27, 'stoch_none_b': 46, 'stoch_none_c': 50, 'stoch_some_a': 78, 'stoch_some_b': 79, 'stoch_some_c': 79, 'stoch_alw_a': 85, 'stoch_alw_b': 92, 'stoch_alw_c': 92, 'D_a': 11, 'D_b': 34, 'D_c': 60, 'C_a': 36, 'C_b': 74, 'C_c': 96, 'd_threshold': 0.6614031559185201, 'c_threshold': 0.43518072565210436}. Best is trial 1 with value: 2.705964285714286.


Best trial: 1. Best value: 2.70596:   1%|          | 3/300 [01:47<3:06:03, 37.59s/it]

...

[I 2026-03-03 19:10:07,748] Trial 299 finished with value: 2.703857142857143 and parameters: {'coop_low_a': 46, 'coop_low_b': 47, 'coop_low_c': 48, 'coop_med_a': 22, 'coop_med_b': 29, 'coop_med_c': 54, 'coop_high_a': 74, 'coop_high_b': 94, 'coop_high_c': 95, 'adap_no_a': 23, 'adap_no_b': 37, 'adap_no_c': 45, 'adap_yes_a': 82, 'adap_yes_b': 84, 'adap_yes_c': 93, 'forg_sigma': 35.35955668880926, 'forg_med_a': 68, 'forg_med_b': 69, 'forg_med_c': 71, 'forg_high_a': 71, 'forg_high_b': 72, 'forg_high_c': 73, 'stoch_none_a': 38, 'stoch_none_b': 39, 'stoch_none_c': 43, 'stoch_some_a': 75, 'stoch_some_b': 77, 'stoch_some_c': 78, 'stoch_alw_a': 81, 'stoch_alw_b': 86, 'stoch_alw_c': 87, 'D_a': 49, 'D_b': 50, 'D_c': 51, 'C_a': 48, 'C_b': 73, 'C_c': 80, 'd_threshold': 0.33694545016644206, 'c_threshold': 0.34318109903755895}. Best is trial 99 with value: 2.7615357142857144.


Best trial: 99. Best value: 2.76154: 100%|██████████| 300/300 [2:30:40<00:00, 30.13s/it]


[I 2026-03-03 19:10:37,087] Trial 300 finished with value: 2.7451785714285712 and parameters: {'coop_low_a': 48, 'coop_low_b': 49, 'coop_low_c': 50, 'coop_med_a': 24, 'coop_med_b': 28, 'coop_med_c': 56, 'coop_high_a': 71, 'coop_high_b': 83, 'coop_high_c': 84, 'adap_no_a': 24, 'adap_no_b': 38, 'adap_no_c': 42, 'adap_yes_a': 54, 'adap_yes_b': 83, 'adap_yes_c': 93, 'forg_sigma': 39.17712314775042, 'forg_med_a': 70, 'forg_med_b': 73, 'forg_med_c': 74, 'forg_high_a': 60, 'forg_high_b': 71, 'forg_high_c': 72, 'stoch_none_a': 40, 'stoch_none_b': 41, 'stoch_none_c': 45, 'stoch_some_a': 55, 'stoch_some_b': 65, 'stoch_some_c': 66, 'stoch_alw_a': 82, 'stoch_alw_b': 86, 'stoch_alw_c': 87, 'D_a': 51, 'D_b': 52, 'D_c': 60, 'C_a': 45, 'C_b': 72, 'C_c': 81, 'd_threshold': 0.36244722974744326, 'c_threshold': 0.3622514248047699}. Best is trial 99 with value: 2.7615357142857144.

=== OPTIMIZATION COMPLETE ===
Best score:  2.7615
Best params: {'coop_low_a': 43, 'coop_low_b': 45, 'coop_low_c': 46, 'coop_me

c:\Users\Ognjen\AppData\Local\Programs\Python\Python311\Lib\site-packages\optuna\importance\_fanova\_tree.py:137: RuntimeWarning: invalid value encountered in divide
  weights = weights / active_features_cardinalities


AssertionError: 